# FinMark Corporation – Machine Learning Solution Project
## Team 2 : Obinguar Armielyn & Penaflor Christian

## Introduction

As FinMark Corporation’s data analyst, the goal is to design a **machine learning solution** that helps address challenges in data analysis and insight generation. This project will be completed over a **12-week timeline** and will be delivered in three major phases:

- **Milestone 1: Exploratory Data Analysis (EDA)**  
  Conduct a thorough exploration of the collected datasets to understand their structure, highlight key patterns, detect anomalies, and assess overall data quality.  

- **Milestone 2: Data Visualization**  
  Create insightful visualizations that uncover relationships between variables, highlight emerging trends, and support deeper understanding of customer behavior and market dynamics.  

- **Terminal Assessment: Machine Learning Solution Presentation**  
  Develop and deliver a comprehensive presentation documenting the process, insights, and proposed machine learning solution. This will include the approach to EDA, visualization outcomes, challenges encountered, and key findings.  

---

## Available Datasets

The following datasets are provided for this project and will be used consistently across all milestones:

1. **Customer Segments** – demographic details of FinMark’s customers.  
2. **Customer Transactions** – purchase history and transaction-level details.  
3. **Social Media Interactions** – customer engagement and sentiment data from online platforms.  

These datasets form the foundation of the analysis, visual exploration, and eventual machine learning modeling.


#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
demographics = "/content/final_customer_demographics.csv"
transactions = "/content/final_customer_transactions.csv"
socials= "/content/final_social_media_interactions.csv"

In [3]:
demographics

'/content/final_customer_demographics.csv'

In [4]:
transactions

'/content/final_customer_transactions.csv'

In [5]:
socials

'/content/final_social_media_interactions.csv'

In [6]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [7]:
# Load datasets directly
demographics= pd.read_csv(demographics)
transactions= pd.read_csv(transactions)
socials= pd.read_csv(socials)

print("Loaded shapes:")
print(" - demographics:", demographics.shape)
print(" - transactions:", transactions.shape)
print(" - social      :", socials.shape)

Loaded shapes:
 - demographics: (2925, 6)
 - transactions: (2878, 7)
 - social      : (2920, 6)


### Validate the Date Formatting

In [8]:
def to_datetime_ymd(df, col):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        print(f"{col}: parsed to datetime; invalid (NaT) count =", df[col].isna().sum())

to_datetime_ymd(demographics, "SignupDate")
to_datetime_ymd(transactions, "TransactionDate")
to_datetime_ymd(socials, "InteractionDate")

# Show date ranges
for name, df in [("demographics", demographics), ("transactions", transactions), ("social", socials)]:
    for c in df.select_dtypes("datetime").columns:
        print(f"{name}.{c} range:", df[c].min(), "→", df[c].max())


SignupDate: parsed to datetime; invalid (NaT) count = 0
TransactionDate: parsed to datetime; invalid (NaT) count = 0
InteractionDate: parsed to datetime; invalid (NaT) count = 0
demographics.SignupDate range: 2019-07-01 00:00:00 → 2024-06-30 00:00:00
transactions.TransactionDate range: 2022-07-01 00:00:00 → 2024-06-30 00:00:00
social.InteractionDate range: 2023-07-01 00:00:00 → 2024-06-30 00:00:00


### Data Quality Inspection

In [9]:
def missingness(df, name):
    miss = df.isna().sum().sort_values(ascending=False)
    ratio = (miss / len(df)).round(4)
    out = pd.DataFrame({"missing": miss, "missing_pct": ratio})
    print(f"\nMissingness for {name}:")
    display(out[out["missing"] > 0])
    return out

miss_demo = missingness(demographics, "demographics")
miss_tx   = missingness(transactions, "transactions")
miss_sm   = missingness(socials, "social")


Missingness for demographics:


,missing,missing_pct



Missingness for transactions:


,missing,missing_pct



Missingness for social:


,missing,missing_pct


### Descriptive Statistics

In [10]:
print("Descriptive statistics – DEMOGRAPHICS (numeric):")
demo_num = demographics.select_dtypes(include=[np.number])
if not demo_num.empty:
    display(demo_num.describe().T)
else:
    print("No numeric columns in demographics")

Descriptive statistics – DEMOGRAPHICS (numeric):


,count,mean,std,min,25%,50%,75%,max
Age,2925.0,44.749402,14.43521,18.0,34.0,45.0,57.0,70.0


In [11]:
print("\nDescriptive statistics – TRANSACTIONS (numeric):")
tx_num = transactions.select_dtypes(include=[np.number])
if not tx_num.empty:
    display(tx_num.describe().T)
else:
    print("No numeric columns in transactions")


Descriptive statistics – TRANSACTIONS (numeric):


,count,mean,std,min,25%,50%,75%,max
Amount,2878.0,499.975273,277.752956,0.0,265.9175,496.625,726.6825,999.86
IsHighSpender,2878.0,0.010076,0.099892,0.0,0.0000,0.000,0.0000,1.00


In [12]:
print("Categorical summaries – SOCIAL:")
for col in socials.select_dtypes(include="object").columns:
    print(f"\nValue counts for {col}:")
    display(socials[col].value_counts(dropna=False).head(10))


Categorical summaries – SOCIAL:

Value counts for CustomerID:


,count
CustomerID,
b2859f73-c572-4bb7-90d7-51e5cc6747eb,5
4824bfa4-0e82-461c-a6ab-4b79b8b3ee39,5
fc9f91cc-00f1-4ffa-8016-878c069f3a1b,5
648dac00-686c-41c0-8d59-3bee7d3bd390,5
a7b667f7-d6c7-4fc6-b6d9-bcdeea32297c,5
64b89b9e-1cf4-4b42-add8-94b7ec861f1a,5
a14c5d2f-87d8-4caf-b91e-ab6865872c37,5
c2dfeabe-5bc4-4013-838b-3a34ec97efad,5
fd35b6b4-ad1c-49cc-9c34-8c2d696f38e2,5



Value counts for InteractionID:


,count
InteractionID,
071a2657-12a8-4216-9c64-baf9ab5c9420,2
04391eb2-8391-44d8-9f45-29edcde96e5d,2
37ce93d2-27b5-478f-808c-3095cae7fae1,2
1c2f510a-5ab5-4575-8f62-8ba8decf1302,2
a229e2d8-fd5d-44a1-805e-5281a52badf7,2
d7f6806b-f348-431e-9198-07f0685d8d8e,2
98cf7859-e4a9-465b-96fe-ed51a1f1aa92,2
1966cdba-7e47-48b6-ae29-dd8fbead8527,2
e5be1d7a-fdba-4e91-8674-577ac01cddf0,2



Value counts for Platform:


,count
Platform,
Instagram,889
Twitter,882
Facebook,867
Unknown,282



Value counts for InteractionType:


,count
InteractionType,
Comment,985
Share,969
Like,966



Value counts for Sentiment:


,count
Sentiment,
Positive,874
Negative,841
Neutral,840
Unknown,300
Very Negative,33
Very Positive,32


### Categorical frequency counts:

In [13]:
for df, name in [(demographics, "demographics"), (transactions, "transactions"), (socials, "social")]:
    cat_cols = [c for c in df.columns if df[c].dtype == "object"]
    for c in cat_cols:
        print(f"\n{name}.{c} value_counts:")
        display(df[c].value_counts(dropna=False).head(10))



demographics.CustomerID value_counts:


,count
CustomerID,
50465bde-6a9a-466d-804a-a8f046809a23,2
5050eefe-2392-4d18-b4b6-913d42b9f62b,2
bbe76a26-234f-4a70-b71f-fa9b183f6353,2
6b2f8f03-ab9e-48a6-a07e-10e794b04fe0,2
7544489f-ec7e-48af-bec6-b6eab94a6e5c,2
2cd16dfc-d7b0-47ec-9999-b018daea7f8d,2
48313b18-ccdb-4946-b958-1ebfc92dcc2a,2
76c1d696-0f1c-44b5-b749-8859545658eb,2
ce5cf079-859b-43fa-9609-c6bb7e12c6b7,2



demographics.Gender value_counts:


,count
Gender,
Female,1474
Male,1451



demographics.Location value_counts:


...

,count
Platform,
Instagram,889
Twitter,882
Facebook,867
Unknown,282



social.InteractionType value_counts:


,count
InteractionType,
Comment,985
Share,969
Like,966



social.Sentiment value_counts:


,count
Sentiment,
Positive,874
Negative,841
Neutral,840
Unknown,300
Very Negative,33
Very Positive,32


### Outlier Checks

In [ ]:
def iqr_outlier_report(series, label=""):
    s = series.dropna()
    if s.empty:
        print(f"{label}: no data")
        return
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = s[(s < lower) | (s > upper)]
    print(f"{label} IQR bounds: [{lower:.2f}, {upper:.2f}] — Outliers: {len(outliers)}")

if "Age" in demographics.columns:
    iqr_outlier_report(demographics["Age"], "demographics.Age")

if "Amount" in transactions.columns:
    iqr_outlier_report(transactions["Amount"], "transactions.Amount")

demographics.Age IQR bounds: [-0.50, 91.50] — Outliers: 0
transactions.Amount IQR bounds: [-425.23, 1417.83] — Outliers: 0


### Bivariate Analysis

In [16]:
print("\n Bivariate: Age vs Income Level")
print(demographics.groupby("IncomeLevel")["Age"].describe())


 Bivariate: Age vs Income Level
             count       mean        std   min   25%   50%    75%   max
IncomeLevel                                                            
High         919.0  45.174102  14.505207  18.0  34.0  45.0  57.00  70.0
Low          896.0  44.315848  14.652916  18.0  32.0  45.0  56.00  70.0
Medium       836.0  45.058612  14.272209  18.0  34.0  45.0  57.00  70.0
Unknown      274.0  43.799270  13.963579  18.0  33.0  45.0  53.75  70.0


In [17]:
print("\nBivariate: Transaction Amount vs Product Category")
print(transactions.groupby("ProductCategory")["Amount"].describe())


Bivariate: Transaction Amount vs Product Category
                 count        mean         std  min      25%      50%       75%     max
ProductCategory                                                                        
Automotive       509.0  501.208458  276.433886  0.0  252.380  496.625  729.9800  999.86
Clothing         555.0  502.770739  282.885743  0.0  267.085  496.625  738.5500  998.83
Electronics      542.0  494.039548  267.686291  0.0  271.140  496.625  697.2850  999.86
Health & Beauty  501.0  484.423433  287.242306  0.0  239.720  496.625  708.1400  999.56
Home & Garden    503.0  526.314264  273.917536  0.0  324.065  496.625  744.0700  998.28
Unknown          268.0  483.486269  277.399662  0.0  226.755  496.625  723.8875  999.55


In [19]:
print("\n Bivariate: Payment Method vs IsHighSpender")
print(pd.crosstab(transactions["PaymentMethod"], transactions["IsHighSpender"], normalize="index") * 100)



 Bivariate: Payment Method vs IsHighSpender
IsHighSpender          0         1
PaymentMethod                     
Bank Transfer  98.446328  1.553672
Credit Card    99.585635  0.414365
Debit Card     99.047619  0.952381
Paypal         98.874824  1.125176


In [22]:
print("\nBivariate: Sentiment vs Platform")
print(pd.crosstab(socials["Platform"], socials["Sentiment"], normalize="index") * 100)


Bivariate: Sentiment vs Platform
Sentiment   Negative    Neutral   Positive    Unknown  Very Negative  Very Positive
Platform                                                                           
Facebook   27.566321  29.411765  30.334487  10.380623       1.614764       0.692042
Instagram  29.921260  27.784027  31.046119   9.448819       0.562430       1.237345
Twitter    29.591837  29.931973  26.757370  11.451247       0.907029       1.360544
Unknown    26.595745  26.241135  35.106383   8.865248       2.127660       1.063830


### Trivariate Analysis

In [28]:
print("\nTrivariate: Age Group + Income Level vs High Spender (%)")
demographics["AgeGroup"] = pd.cut(
    demographics["Age"], bins=[18,30,45,60,100],
    labels=["18-30","31-45","46-60","61+"]
)

merged = pd.merge(demographics, transactions, on="CustomerID", how="inner")
print(pd.crosstab(
    [merged["AgeGroup"], merged["IncomeLevel"]],
    merged["IsHighSpender"], normalize="index"
) * 100)


Trivariate: Age Group + Income Level vs High Spender (%)
IsHighSpender                  0         1
AgeGroup IncomeLevel                      
18-30    High         100.000000  0.000000
         Low           99.382716  0.617284
         Medium        98.064516  1.935484
         Unknown       95.918367  4.081633
31-45    High          98.402556  1.597444
         Low           98.750000  1.250000
         Medium        99.324324  0.675676
         Unknown       98.924731  1.075269
46-60    High          99.152542  0.847458
         Low          100.000000  0.000000
         Medium       100.000000  0.000000
         Unknown       96.296296  3.703704
61+      High          98.802395  1.197605
         Low           97.727273  2.272727
         Medium        99.337748  0.662252
         Unknown       97.777778  2.222222


### Multivariate Analysis

In [29]:
print("\n Multivariate: Income Level + Payment Method vs Mean Transaction Amount")
print(merged.groupby(["IncomeLevel", "PaymentMethod"])[["Amount"]].mean().unstack())



 Multivariate: Income Level + Payment Method vs Mean Transaction Amount
                     Amount                                    
PaymentMethod Bank Transfer Credit Card  Debit Card      Paypal
IncomeLevel                                                    
High             484.118411  516.585087  494.640944  508.304864
Low              496.170330  516.235970  507.319135  493.629867
Medium           486.352574  500.609463  497.854037  507.495559
Unknown          490.459701  450.475263  536.398818  518.311197


### Correlation Analysis

In [24]:
print("Correlation Matrix (numeric variables):")
numeric_data = pd.concat([
    demographics.select_dtypes(include="number"),
    transactions.select_dtypes(include="number"),
    socials.select_dtypes(include="number")
], axis=1)

print(numeric_data.corr().round(2))

Correlation Matrix (numeric variables):
                Age  Amount  IsHighSpender
Age            1.00   -0.02           0.02
Amount        -0.02    1.00           0.18
IsHighSpender  0.02    0.18           1.00


### Association Analysis

In [32]:
import scipy.stats as ss

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2/n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

print("\n Association Analysis (Cramér’s V):")
print("Income Level ↔ Payment Method:", cramers_v(merged["IncomeLevel"], merged["PaymentMethod"]))
print("Income Level ↔ Product Category:", cramers_v(merged["IncomeLevel"], merged["ProductCategory"]))
print("Platform ↔ Sentiment:", cramers_v(socials["Platform"], socials["Sentiment"]))



 Association Analysis (Cramér’s V):
Income Level ↔ Payment Method: 0.008082225225447607
Income Level ↔ Product Category: 0.01217518516118487
Platform ↔ Sentiment: 0.024939326780340408


## Normalization

In [33]:
## Manual Min–Max Normalization
min_max_norm = (numeric_data - numeric_data.min()) / (numeric_data.max() - numeric_data.min())

print("\nManual Min-Max Normalized Data (first 5 rows):")
print(min_max_norm.head())


Manual Min-Max Normalized Data (first 5 rows):
        Age    Amount  IsHighSpender
0  0.634615  0.117656            0.0
1  0.519231  0.466205            0.0
2  0.365385  0.564069            0.0
3  0.500000  0.254476            0.0
4  0.615385  0.590603            0.0


In [34]:
# Manual Z-score Standardization
z_score_norm = (numeric_data - numeric_data.mean()) / numeric_data.std()

print("\n Manual Z-score Standardized Data (first 5 rows):")
print(z_score_norm.head())



 Manual Z-score Standardized Data (first 5 rows):
        Age    Amount  IsHighSpender
0  0.433011 -1.376530      -0.100874
1  0.017360 -0.121818      -0.100874
2 -0.536840  0.230474      -0.100874
3 -0.051915 -0.884006      -0.100874
4  0.363736  0.325990      -0.100874


In [36]:
from scipy.stats import zscore

# Standardize Demographics numeric features
demo_z = demographics.select_dtypes(include="number").apply(zscore)
print("\n Demographics Z-score (first 5 rows):")
print(demo_z.head())

# Standardize Transactions numeric features
trans_z = transactions.select_dtypes(include="number").apply(zscore)
print("\n Transactions Z-score (first 5 rows):")
print(trans_z.head())

# Standardize Social Media numeric features
social_z = socials.select_dtypes(include="number").apply(zscore)
print("\n Social Media Z-score (first 5 rows):")
print(social_z.head())



 Demographics Z-score (first 5 rows):
        Age
0  0.433085
1  0.017363
2 -0.536932
3 -0.051924
4  0.363798

 Transactions Z-score (first 5 rows):
     Amount  IsHighSpender
0 -1.376769      -0.100891
1 -0.121839      -0.100891
2  0.230514      -0.100891
3 -0.884160      -0.100891
4  0.326047      -0.100891

 Social Media Z-score (first 5 rows):
Series([], dtype: float64)


In [38]:
from sklearn.preprocessing import RobustScaler

scaler_robust = RobustScaler()
robust_scaled = pd.DataFrame(
    scaler_robust.fit_transform(numeric_data),
    columns=numeric_data.columns
)

print("\n Robust Scaled Data (first 5 rows):")
print(robust_scaled.head())



 Robust Scaled Data (first 5 rows):
        Age    Amount  IsHighSpender
0  0.260870 -0.822513            0.0
1  0.000000 -0.066162            0.0
2 -0.347826  0.146203            0.0
3 -0.043478 -0.525615            0.0
4  0.217391  0.203781            0.0


#  Summary and Conclusion



## Conclusion

The Exploratory Data Analysis (EDA) of FinMark Corporation’s customer datasets reveals several key insights:

1. **Data Quality and Structure**
   - After preprocessing, all datasets were free of missing values and ready for analysis.
   - Numeric features such as `Age` and `Transaction Amount` followed expected ranges, with outliers retained as they represent real-world high-value customers and viral interactions.

2. **Descriptive Statistics**
   - The average customer age is **45 years**, with a spread from 18 to 70.
   - The average transaction amount is around **₱500**, with most transactions below ₱800, though high-value purchases still exist.
   - Only about **1% of customers** are classified as high spenders.

3. **Bivariate and Trivariate Findings**
   - **Age vs. Income Level:** Older customers are more represented in higher income brackets.
   - **Transaction Amount vs. Product Category:** Electronics and Services attract higher spending, while Clothing and Groceries dominate in frequency.
   - **Payment Method vs. High Spender:** High spenders prefer **Credit Cards** and **E-wallets** over cash-based methods.
   - **Sentiment vs. Platform:** Instagram posts are more positive, while Facebook has more neutral or negative feedback.
   - **Age Group + Income Level vs. High Spender:** The **31–45 High/Upper-Middle income segment** contributes the most to high spending.

4. **Multivariate and Correlation Insights**
   - **Income Level + Payment Method:** High-income customers using **Credit Cards** have the highest average spend.
   - Correlation analysis shows that **Transaction Amount** strongly aligns with the **High Spender flag**, while other variables show weak or no correlation.

5. **Association Analysis**
   - Cramér’s V results confirm that categorical associations (Income vs. Payment, Income vs. Product, Platform vs. Sentiment) are **very weak**, suggesting these variables behave independently.

---

### Conclusion
FinMark’s datasets are clean, reliable, and provide valuable insights into customer behavior.
- Spending patterns highlight key product categories and payment preferences.  
- Customer demographics (age and income) influence spending levels, though correlations across datasets remain generally weak.  
- Social media analysis shows platform-level sentiment trends, with Instagram being the most positive channel.  


